In [1]:
include("../RayTracing.jl")

Main.RayTracing

# Calculating intersection & normals with PBRT

### Case 1: hitting the closest point

In [2]:
center = RayTracing.Pnt3(1,1,1)
radius = 2.0

ray = RayTracing.Ray(
    RayTracing.Pnt3(10, 10, 10),
    RayTracing.Vec3(-1, -1, -1),
    0.0,
    999999999.0
)

sphere_transform = RayTracing.Translate(center)
sphere = RayTracing.Sphere(
    RayTracing.ShapeCore(
        sphere_transform,
        RayTracing.Inv(sphere_transform),
        false,
        false
    ),
    radius
)

check, t, si = RayTracing.intersect(sphere, ray)

@assert si.core.p ≈ RayTracing.Pnt3(2.1547005383792515, 2.1547005383792515, 2.1547005383792515)
@assert si.core.n ≈ RayTracing.Pnt3(0.5773502691896258, 0.5773502691896258, 0.5773502691896258)

### Case 2: hitting the 'corner' at x,y,0

In [3]:
center = RayTracing.Pnt3(1,1,1)
radius = 2.0

ray = RayTracing.Ray(
    RayTracing.Pnt3(10, 10, 10),
    RayTracing.Vec3(sqrt((radius^2)/2)+center.x, sqrt((radius^2)/2)+center.y, center.z) - RayTracing.Pnt3(10, 10, 10),
    0.0,
    999999999.0
)

sphere_transform = RayTracing.Translate(center)
sphere = RayTracing.Sphere(
    RayTracing.ShapeCore(
        sphere_transform,
        RayTracing.Inv(sphere_transform),
        false,
        false
    ),
    radius
)

check, t, si = RayTracing.intersect(sphere, ray)

@assert si.core.p ≈ RayTracing.Pnt3(2.414213562373095, 2.414213562373095, 1.0)
@assert si.core.n ≈ RayTracing.Pnt3(0.7071067811865476, 0.7071067811865476, 0)

# OK! Now to confirm the normal math works!

In [4]:
function f(s::RayTracing.Sphere, p::RayTracing.Pnt3)::Float64
    center = s.core.object_to_world(RayTracing.Pnt3(0,0,0))
    return RayTracing.distance_squared(center, p) - s.radius^2
end

function normal_slow(s::RayTracing.Sphere, p::RayTracing.Pnt3)::RayTracing.Vec3
    h = 0.00001
    fx = (f(s, p + RayTracing.Pnt3(h,0,0)) - f(s, p + RayTracing.Pnt3(-h,0,0)))/(2.0 * h)
    fy = (f(s, p + RayTracing.Pnt3(0,h,0)) - f(s, p + RayTracing.Pnt3(0,-h,0)))/(2.0 * h)
    fz = (f(s, p + RayTracing.Pnt3(0,0,h)) - f(s, p + RayTracing.Pnt3(0,0,-h)))/(2.0 * h)
    return RayTracing.normalize(RayTracing.Vec3(fx, fy, fz))
end

function normal(s::RayTracing.Sphere, p::RayTracing.Pnt3)::RayTracing.Vec3
    e = .00001
    return RayTracing.normalize(
        RayTracing.Vec3(1, -1, -1) * f(s, p + RayTracing.Vec3(e, -e, -e)) +
        RayTracing.Vec3(-1, -1, 1) * f(s, p + RayTracing.Vec3(-e, -e, e)) +
        RayTracing.Vec3(-1, 1, -1) * f(s, p + RayTracing.Vec3(-e, e, -e)) +
        RayTracing.Vec3(1, 1, 1) * f(s, p + RayTracing.Vec3(e, e, e))
    )
end

normal (generic function with 1 method)

In [5]:
@assert normal_slow(sphere, RayTracing.Pnt3(2.1547005383792515, 2.1547005383792515, 2.1547005383792515)) ≈ RayTracing.Vec3(0.5773502691896258, 0.5773502691896258, 0.5773502691896258)
@assert normal_slow(sphere, RayTracing.Pnt3(2.414213562373095, 2.414213562373095, 1.0)) ≈ RayTracing.Vec3(0.7071067811865476, 0.7071067811865476, 0)

@assert normal(sphere, RayTracing.Pnt3(2.1547005383792515, 2.1547005383792515, 2.1547005383792515)) ≈ RayTracing.Vec3(0.5773502691896258, 0.5773502691896258, 0.5773502691896258)
@assert normal(sphere, RayTracing.Pnt3(2.414213562373095, 2.414213562373095, 1.0)) ≈ RayTracing.Vec3(0.7071067811865476, 0.7071067811865476, 0)

# Now, let's see if we can back into dpdu and dpdv and dndu dndv

In [6]:
n, v1, v2 = RayTracing.orthonormal_basis(RayTracing.Vec3(si.core.n))

([0.7071067811865476, 0.7071067811865476, 5.551115123104666e-12], [0.0, 7.850462293389012e-12, -1.0], [-0.7071067811865476, 0.7071067811865476, 5.551115123104666e-12])

# OK now to add soft bodies :)

## Step 1 find roots

In [7]:
using Roots

In [8]:
struct SoftBody
    ks::Vector{RayTracing.Pnt3}
    R::Float64
    magic::Float64
end

In [47]:
# the answers I got from wolfram
ts = [9.13397, 10.866]

2-element Vector{Float64}:
  9.13397
 10.866

In [13]:
# for use in intersection point calc
# given ray & t
function f(soft_body::SoftBody, t::Float64, ray::RayTracing.Ray)::Float64
    f_val = 0.0 - soft_body.magic # start at negative target so we can "solve for zero"
    for k in soft_body.ks
        p = RayTracing.norm(RayTracing.at(ray, t)-k)
        if p <= soft_body.R
            f_val += -.4444 * (p ^ 6) / (R ^ 6) + 1.8888 * (p ^ 4) / (R ^ 4) - 2.4444 * (p ^ 2) / (R ^ 2) + 1.0
        end
    end
    return f_val
end

# for use in normal calc
function f(soft_body::SoftBody, pp::RayTracing.Pnt3)::Float64
    f_val = 0.0 - soft_body.magic # do I need target here?
    for k in soft_body.ks
        p = RayTracing.norm(pp-k)
        if p <= soft_body.R
            f_val += -.4444 * (p ^ 6) / (R ^ 6) + 1.8888 * (p ^ 4) / (R ^ 4) - 2.4444 * (p ^ 2) / (R ^ 2) + 1.0
        end
    end
    return f_val
end

f (generic function with 3 methods)

In [14]:
# instantiate stuff
softy = SoftBody(
    RayTracing.Pnt3[RayTracing.Pnt3(0,0,0)],
    3.0,
    0.5
)
ray = RayTracing.Ray(RayTracing.Pnt3(10, 10, 10), RayTracing.Vec3(1, 1.1, 1), 0.0, 999999999.0)

# creating my anonymous function
# https://stackoverflow.com/questions/53824498/finding-univariate-roots-in-julia-of-a-function-with-many-arguments
tmp_solve = (x -> f(softy, x, ray))

# solve
solutions = find_zeros(tmp_solve, 0, 20)

# assert
# @assert sum(ts - zeros) < .001

t = minimum(solutions)

f(softy, t, ray)

MethodError: MethodError: reducing over an empty collection is not allowed; consider supplying `init` to the reducer

In [43]:
function normal(soft_body::SoftBody, p::RayTracing.Pnt3)::RayTracing.Vec3
    e = .00001
    return RayTracing.normalize(
        RayTracing.Vec3(1, -1, -1) * f(soft_body, p + RayTracing.Vec3(e, -e, -e)) +
        RayTracing.Vec3(-1, -1, 1) * f(soft_body, p + RayTracing.Vec3(-e, -e, e)) +
        RayTracing.Vec3(-1, 1, -1) * f(soft_body, p + RayTracing.Vec3(-e, e, -e)) +
        RayTracing.Vec3(1, 1, 1)   * f(soft_body, p + RayTracing.Vec3(e, e, e))
    )
end

normal (generic function with 1 method)

In [52]:
normal(softy, RayTracing.at(ray, t))

3-element Main.RayTracing.Vec3 with indices SOneTo(3):
 -0.7030692303193209
 -0.10671136188989208
 -0.7030692303193209